# LeetCode #1124 — Longest Well-Performing Interval

**Medium · Python 3 · Array / Prefix Sum / Hash Map / Monotonic Stack**  
[Đề gốc LeetCode](https://leetcode.com/problems/longest-well-performing-interval/) · [Đối chiếu đề, ví dụ và constraints](https://leetcode.doocs.org/en/lc/1124/)

```python
class Solution:
    def longestWPI(self, hours: List[int]) -> int:
        ...
```

**Mục tiêu:** học cách biến câu hỏi “số ngày mệt nhiều hơn ngày không mệt” thành **tổng dương trên một đoạn**, rồi dùng Hash Map giảm từ `O(n²)` xuống `O(n)` trung bình. Không cần biết trước `dict`, Prefix Sum hoặc Big-O.

## 1. Mục tiêu cụ thể

Sau bài này bạn có thể:

1. Dùng `> 8` chính xác, kể cả khi ngày làm **đúng 8 giờ**.
2. Tự kiểm tra một đoạn `[l..r]` có *well-performing* hay không.
3. Giải thích được vì sao đổi giờ làm thành `+1` và `-1`.
4. Hiểu công thức `prefix[j] - prefix[i]` và lý do lưu **vị trí đầu tiên** của một tổng.
5. Chứng minh vì sao chỉ cần tra `score - 1`, không phải mọi tổng nhỏ hơn `score`.
6. Viết và kiểm thử `Solution.longestWPI` đúng kiểu LeetCode.

## 2. Đề bài, ví dụ và giới hạn chính xác

`hours[i]` là số giờ làm việc ngày thứ `i`.

- Ngày **mệt** (*tiring*) khi và chỉ khi `hours[i] > 8`.
- Ngày **không mệt** khi `hours[i] <= 8`.
- Một đoạn ngày **liên tiếp, không rỗng** đạt yêu cầu nếu **số ngày mệt > số ngày không mệt**.
- Trả về **độ dài lớn nhất**; nếu không có đoạn nào đạt thì trả `0`.

| Đầu vào | Đáp án | Giải thích |
|---|---:|---|
| `[9,9,6,0,6,6,9]` | `3` | Đoạn `[9,9,6]`: 2 ngày mệt, 1 ngày không mệt |
| `[6,6,6]` | `0` | Không có ngày mệt |

**Constraints:** `1 <= hours.length <= 10^4`; `0 <= hours[i] <= 16`.

## 3. Nền tảng Python: so sánh, vòng lặp, chỉ số

Python đánh chỉ số từ `0`. `enumerate(hours)` lần lượt cho cặp `(i, x)` gồm **vị trí** và **giá trị**. Toán tử `>` là *lớn hơn nghiêm ngặt*, không giống `>=`.

Ví dụ `hours=[8,9,7]`: chỉ `9` là ngày mệt.

In [1]:
hours_demo = [8, 9, 7]
for i, x in enumerate(hours_demo):
    print('Ngày', i, '| giờ =', x, '| mệt?', x > 8)
assert [x > 8 for x in hours_demo] == [False, True, False]

Ngày 0 | giờ = 8 | mệt? False
Ngày 1 | giờ = 9 | mệt? True
Ngày 2 | giờ = 7 | mệt? False


## 4. Thử đếm trực tiếp một đoạn

Với `hours=[9,9,6,0,6,6,9]`, đoạn từ `l=0` đến `r=2` chứa 3 ngày: `[9,9,6]`.

- Số ngày mệt = 2.
- Số ngày không mệt = 1.
- `2 > 1` nên đoạn hợp lệ.

`hours[l:r+1]` nghĩa là lấy từ vị trí `l` đến **trước** `r+1`, tức bao gồm `r`.

In [2]:
hours_demo = [9,9,6,0,6,6,9]
l, r = 0, 2
part = hours_demo[l:r+1]
tiring = sum(x > 8 for x in part)  # True được tính như 1
non_tiring = len(part) - tiring
print(part, 'mệt:', tiring, 'không mệt:', non_tiring)
assert (tiring, non_tiring) == (2,1)

[9, 9, 6] mệt: 2 không mệt: 1


## 5. Cách trực giác — thử mọi đoạn (Brute Force)

Mỗi đoạn được xác định bởi hai vị trí `l <= r`. Giữ `l` cố định, mở rộng `r` sang phải và cập nhật số ngày mệt/không mệt. Khi `mệt > không mệt`, xét độ dài `r-l+1`.

**Lưu ý:** nếu dùng `sum(...)` lại cho *mỗi* đoạn thì có thể tốn `O(n³)`. Bản dưới cộng dồn khi mở rộng đoạn nên chỉ còn `O(n²)` thời gian, `O(1)` bộ nhớ phụ.

In [3]:
def longest_wpi_bruteforce(hours):
    best = 0
    for l in range(len(hours)):
        score = 0
        for r in range(l, len(hours)):
            score += 1 if hours[r] > 8 else -1
            if score > 0:
                best = max(best, r - l + 1)
    return best

for arr in ([9,9,6,0,6,6,9], [6,6,6], [9], [8,9]):
    print(arr, '->', longest_wpi_bruteforce(arr))
assert longest_wpi_bruteforce([9,9,6,0,6,6,9]) == 3
assert longest_wpi_bruteforce([6,6,6]) == 0

[9, 9, 6, 0, 6, 6, 9] -> 3
[6, 6, 6] -> 0
[9] -> 1
[8, 9] -> 1


## 6. Bước đột phá: đổi mỗi ngày thành `+1` hoặc `-1`

Một đoạn đạt yêu cầu khi:

`số ngày mệt - số ngày không mệt > 0`.

Vậy ta gán mỗi ngày:

- `+1` nếu giờ `>8`,
- `-1` nếu giờ `<=8`.

**Tổng của đoạn dương ⇔ đoạn well-performing.** Đây là sự tương đương toán học, không phải mẹo đoán đáp án.

In [4]:
hours_demo = [9,9,6,0,6,6,9]
changes = [1 if x > 8 else -1 for x in hours_demo]
print('Giờ:     ', hours_demo)
print('Đổi dấu: ', changes)
assert changes == [1,1,-1,-1,-1,-1,1]
assert sum(changes[:3]) == 1

Giờ:      [9, 9, 6, 0, 6, 6, 9]
Đổi dấu:  [1, 1, -1, -1, -1, -1, 1]


## 7. Prefix Sum (tổng tiền tố) là gì?

`prefix[k]` là tổng **k phần tử đầu tiên** của mảng `changes`, với `prefix[0]=0`.

`prefix[k+1] = prefix[k] + changes[k]`.

Khi muốn tính đoạn `changes[l..r]`, không cộng lại từng phần tử:

`sum(changes[l:r+1]) = prefix[r+1] - prefix[l]`.

Do đó cần chọn `i = l`, `j = r+1` sao cho **`prefix[j] > prefix[i]`** và `j-i` lớn nhất.

In [5]:
changes = [1,1,-1,-1,-1,-1,1]
prefix = [0]
for value in changes:
    prefix.append(prefix[-1] + value)
print('Chỉ số tiền tố:', list(range(len(prefix))))
print('Giá trị tiền tố:', prefix)
assert prefix == [0,1,2,1,0,-1,-2,-1]
assert prefix[3] - prefix[0] == 1  # 3 ngày đầu

Chỉ số tiền tố: [0, 1, 2, 3, 4, 5, 6, 7]
Giá trị tiền tố: [0, 1, 2, 1, 0, -1, -2, -1]


## 8. Python nền tảng: `dict` lưu vị trí đầu tiên

Dictionary là bảng ánh xạ **khóa → giá trị**. Ví dụ `first[-1] = 4` nghĩa là tổng tiền tố `-1` xuất hiện lần đầu ở vị trí 4 (chỉ số 0-based trong quy ước đang dùng).

Các thao tác quan trọng:

- `{}` tạo dictionary rỗng;
- `key in first` kiểm tra có khóa chưa (trung bình `O(1)`);
- `first[key] = i` lưu vị trí;
- **chỉ lưu khi khóa chưa có** để giữ vị trí sớm nhất, tạo được đoạn dài nhất.

In [6]:
first = {}
for i, score in enumerate([-1, -2, -1, 0]):
    if score not in first:
        first[score] = i
    print('i =', i, '| score =', score, '| first =', first)
assert first[-1] == 0  # không được cập nhật thành 2

i = 0 | score = -1 | first = {-1: 0}
i = 1 | score = -2 | first = {-1: 0, -2: 1}
i = 2 | score = -1 | first = {-1: 0, -2: 1}
i = 3 | score = 0 | first = {-1: 0, -2: 1, 0: 3}


## 9. Insight quan trọng: vì sao chỉ cần `score - 1`?

Sau ngày `i`, gọi `score` là tổng từ ngày 0 đến i.

- Nếu `score > 0`, cả đoạn từ đầu dài `i+1` đã hợp lệ: **không thể có đoạn nào dài hơn kết thúc ở i**.
- Nếu `score <= 0`, ta cần một tổng tiền tố trước đó **nhỏ hơn score**.

Mỗi bước tiền tố đổi **đúng `+1` hoặc `-1`**. Nếu tồn tại một tiền tố cũ `p < score` thì để đi từ `0` đến `p` (với `score<=0`), dãy phải từng chạm `score-1`. Vị trí **đầu tiên** chạm `score-1` không muộn hơn vị trí đang xét với giá trị thấp hơn nữa.

Vì vậy chỉ tìm `first[score-1]` là đủ để đạt độ dài tối đa ở vị trí cuối i. Đây là tính chất chỉ đúng khi các bước là `±1`; không tùy tiện áp dụng nếu các bước có độ lớn khác.

## 10. Lời giải tối ưu `O(n)` trung bình

Từ trái sang phải, luôn giữ:

- `score`: tổng tiền tố **sau khi xử lý phần tử hiện tại**;
- `first`: chỉ số đầu tiên của mỗi `score` đã xuất hiện;
- `best`: độ dài hợp lệ lớn nhất đã phát hiện.

Thứ tự trong vòng lặp: (1) cộng ±1 → (2) tìm đáp án → (3) chỉ lưu score nếu chưa có. Nhớ phân biệt `i` (chỉ số ngày) và `i+1` (số ngày từ đầu).

In [7]:
def longest_wpi_hashmap(hours):
    first = {}
    score = 0
    best = 0
    for i, x in enumerate(hours):
        score += 1 if x > 8 else -1
        if score > 0:
            best = i + 1
        elif score - 1 in first:
            best = max(best, i - first[score - 1])
        if score not in first:
            first[score] = i
    return best

for arr in ([9,9,6,0,6,6,9], [6,6,6], [9], [8], [8,9,9]):
    print(arr, '->', longest_wpi_hashmap(arr))
assert longest_wpi_hashmap([9,9,6,0,6,6,9]) == 3

[9, 9, 6, 0, 6, 6, 9] -> 3
[6, 6, 6] -> 0
[9] -> 1
[8] -> 0
[8, 9, 9] -> 3


## 11. Mô phỏng bằng tay: input `[9,9,6,0,6,6,9]`

| i | giờ | ±1 | score sau cập nhật | tìm gì? | best |
|---:|---:|---:|---:|---|---:|
| 0 | 9 | +1 | 1 | `score>0` ⇒ đoạn `[0..0]` | 1 |
| 1 | 9 | +1 | 2 | `score>0` ⇒ đoạn `[0..1]` | 2 |
| 2 | 6 | −1 | 1 | `score>0` ⇒ đoạn `[0..2]` | **3** |
| 3 | 0 | −1 | 0 | tìm `first[-1]` (chưa có) | 3 |
| 4 | 6 | −1 | −1 | tìm `first[-2]` (chưa có) | 3 |
| 5 | 6 | −1 | −2 | tìm `first[-3]` (chưa có) | 3 |
| 6 | 9 | +1 | −1 | tìm `first[-2]=5`, đoạn dài 1 | 3 |

Kết quả 3. **Đặc biệt:** ở `i=2` ta đã ghi `first[1]=0`, nên khi `score` quay lại 1 tại i=2, **không được ghi đè** vị trí 0.

In [8]:
def trace_longest_wpi(hours):
    first, score, best = {}, 0, 0
    print('i | giờ | ±1 | score | hành động | best | first trước khi lưu')
    for i, x in enumerate(hours):
        delta = 1 if x > 8 else -1
        score += delta
        if score > 0:
            best = i + 1
            action = 'lấy cả tiền tố'
        elif score - 1 in first:
            best = max(best, i - first[score - 1])
            action = 'tra first['+str(score-1)+']'
        else:
            action = 'chưa có '+str(score-1)
        print(i, '|', x, '|', delta, '|', score, '|', action, '|', best, '|', dict(first))
        if score not in first:
            first[score] = i
    return best

assert trace_longest_wpi([9,9,6,0,6,6,9]) == 3

i | giờ | ±1 | score | hành động | best | first trước khi lưu
0 | 9 | 1 | 1 | lấy cả tiền tố | 1 | {}
1 | 9 | 1 | 2 | lấy cả tiền tố | 2 | {1: 0}
2 | 6 | -1 | 1 | lấy cả tiền tố | 3 | {1: 0, 2: 1}
3 | 0 | -1 | 0 | chưa có -1 | 3 | {1: 0, 2: 1}
4 | 6 | -1 | -1 | chưa có -2 | 3 | {1: 0, 2: 1, 0: 3}
5 | 6 | -1 | -2 | chưa có -3 | 3 | {1: 0, 2: 1, 0: 3, -1: 4}
6 | 9 | 1 | -1 | tra first[-2] | 3 | {1: 0, 2: 1, 0: 3, -1: 4, -2: 5}


## 12. Phiên bản nộp LeetCode

Trên LeetCode, **chỉ nộp class `Solution`**. Không nộp bảng trace, demo `print`, hàm brute force, hay nội dung bài tập. Bài này trả về **một số nguyên**, không phải chính đoạn con.

In [9]:
from typing import List

class Solution:
    def longestWPI(self, hours: List[int]) -> int:
        first = {}
        score = 0
        ans = 0
        for i, h in enumerate(hours):
            score += 1 if h > 8 else -1
            if score > 0:
                ans = i + 1
            elif score - 1 in first:
                ans = max(ans, i - first[score - 1])
            if score not in first:
                first[score] = i
        return ans

assert Solution().longestWPI([9,9,6,0,6,6,9]) == 3
assert Solution().longestWPI([6,6,6]) == 0
print('Code LeetCode — ví dụ đề: PASS')

Code LeetCode — ví dụ đề: PASS


## 13. Vì sao thuật toán đúng? (invariant)

Sau khi xử lý i, `first[t]` luôn là **vị trí sớm nhất** trong các chỉ số `0..i` có tổng tiền tố bằng `t`. Vì chỉ thêm khóa khi chưa có, tính chất này không bao giờ bị phá.

Xét mọi đoạn hợp lệ kết thúc tại i:

- Nếu tổng tiền tố hiện tại > 0, đoạn bắt đầu ở 0 là dài nhất có thể.
- Nếu tổng <= 0, một đoạn hợp lệ phải bắt đầu sau tiền tố có tổng nhỏ hơn `score`. Do các bước là ±1, tiền tố đầu tiên đạt `score-1` nằm không muộn hơn mọi tiền tố nhỏ hơn score, nên tra `score-1` đủ chọn đoạn dài nhất.

Lấy `max` qua mọi i, ta tìm được đoạn hợp lệ dài nhất toàn mảng.

## 14. Time / Space Complexity

Gọi `n = len(hours)`.

| Cách | Thời gian | Bộ nhớ phụ |
|---|---|---|
| Thử mọi đoạn, cập nhật score | `O(n²)` | `O(1)` |
| Prefix Sum + Hash Map | **`O(n)` trung bình** | `O(n)` |

Mỗi ngày được xử lý một lần. Tra/thêm khóa nguyên trong dictionary mất trung bình `O(1)`, không bảo đảm tuyệt đối trong mọi trường hợp hash table. `first` có tối đa `n` khóa. Nút thắt lớn nhất của brute force là thử cùng một phần tử trong nhiều đoạn khác nhau.

## 15. Bộ test: mẫu đề, biên, phản ví dụ

Để phát hiện lỗi `>= 8`, thử `[8]`. Để phát hiện lỗi trả index thay vì length, thử `[9]`. Để phân biệt “hơn” với “bằng”, thử `[9,8]` (1 ngày mệt, 1 ngày không mệt ⇒ cả đoạn không hợp lệ).

In [10]:
tests = [
    ([9,9,6,0,6,6,9],3),
    ([6,6,6],0),
    ([8],0),
    ([9],1),
    ([9,8],1),
    ([8,9],1),
    ([8,9,9],3),
    ([9,8,9,8,9],5),
    ([8,8,8,8,9],1),
    ([9,9,9,9],4),
    ([0,16,0,16],3),
]
for arr, expected in tests:
    actual = Solution().longestWPI(arr)
    assert actual == expected, (arr, actual, expected)
    assert actual == longest_wpi_bruteforce(arr)
print('Test mẫu và biên:',len(tests),'PASS')

Test mẫu và biên: 11 PASS


## 16. Kiểm thử đối chiếu ngẫu nhiên (oracle)

**Oracle** là lời giải chậm nhưng dễ tin và dễ đọc — ở đây là `longest_wpi_bruteforce`. Chạy hàng trăm mảng nhỏ sinh ngẫu nhiên, so kết quả của bản tối ưu với oracle. Dùng seed để mỗi lần mở notebook đều kiểm tra cùng bộ dữ liệu.

In [11]:
import random
rng = random.Random(1124)
for _ in range(250):
    size = rng.randint(1, 22)
    arr = [rng.randint(0,16) for _ in range(size)]
    assert longest_wpi_hashmap(arr) == longest_wpi_bruteforce(arr), arr
    assert Solution().longestWPI(arr) == longest_wpi_bruteforce(arr), arr
print('250 bộ ngẫu nhiên: PASS')
assert 0 <= Solution().longestWPI([9,6]*5000) <= 10000
print('Input độ dài 10.000: PASS')

250 bộ ngẫu nhiên: PASS
Input độ dài 10.000: PASS


## 17. Cách tổng quát hơn: monotonic stack trên Prefix Sum (tùy chọn)

Cách Hash Map dùng tính chất **mỗi ngày đổi ±1**. Nếu tương lai bài hỏi tìm đoạn có *tổng trọng số bất kỳ* lớn hơn 0, thủ thuật `score - 1` không còn đúng.

Dạng tổng quát vẫn có lời giải `O(n)` bằng:

1. Tạo mảng `prefix` độ dài `n+1`.
2. Đưa các vị trí có tiền tố **nhỏ hơn mọi tiền tố trước** vào stack (danh sách chỉ số, giá trị prefix giảm nghiêm ngặt).
3. Quét điểm phải `j` từ cuối về đầu; nếu `prefix[j] > prefix[stack[-1]]`, cập nhật `j-stack[-1]` và pop. Bởi j đang giảm, vị trí này không thể tạo đoạn dài hơn sau đó.

Khác với **ngăn xếp lời gọi hàm**, stack này chỉ dùng `list.append`, `list.pop`, `list[-1]`.

In [12]:
def max_positive_subarray(values):
    p = [0]
    for x in values:
        p.append(p[-1] + x)
    stack = []
    for i, value in enumerate(p):
        if not stack or value < p[stack[-1]]:
            stack.append(i)
    ans = 0
    for j in range(len(values), -1, -1):
        while stack and p[j] > p[stack[-1]]:
            ans = max(ans, j - stack.pop())
    return ans

assert max_positive_subarray([1,1,-1,-1,-1,-1,1]) == 3
assert max_positive_subarray([-3,2]) == 1  # không được tra score-1 tùy tiện
assert max_positive_subarray([-5,10,-3,2]) == 4
print('Cách monotonic stack tổng quát: PASS')

Cách monotonic stack tổng quát: PASS


## 18. Lỗi thường gặp: code sai → giải thích → sửa

**Lỗi A:** `hours[i] >= 8` — sai do đúng 8 giờ **không mệt**. Sửa thành `>8`.

**Lỗi B:** cho phép `score == 0` — sai, vì số ngày mệt và không mệt bằng nhau. Phải là `score > 0`.

**Lỗi C:** gán `first[score]=i` mỗi lần — ghi đè chỉ số sớm nhất, có thể làm đoạn tìm được ngắn đi.

**Lỗi D:** trả `i-first[score-1]+1` — thừa 1 vì điểm trái là tiền tố tại ngày `j` nên đoạn bắt đầu ở `j+1`; độ dài từ `j+1` tới `i` là `i-j`.

In [13]:
bad_example = [8,9]
wrong_changes = [1 if h >= 8 else -1 for h in bad_example]
correct_changes = [1 if h > 8 else -1 for h in bad_example]
print('Sai >= 8:', wrong_changes, '  Đúng > 8:', correct_changes)
assert wrong_changes != correct_changes
assert Solution().longestWPI(bad_example) == 1

Sai >= 8: [1, 1]   Đúng > 8: [-1, 1]


## 19. Mẹo nhận diện và biến thể

Khi đề có cụm “**trong một đoạn liên tiếp**, loại A xuất hiện **nhiều hơn** loại B”, hãy thử gán `A→+1`, `B→-1` rồi hỏi về **tổng đoạn**. Nếu đề hỏi đoạn có tổng `>0`, biến đổi này đặc biệt hữu ích.

Biến thể tốt để tập: “đếm số ngày mệt trừ ngày không mệt bằng `k`” ⇒ tìm `prefix[j]-prefix[i]=k`; khi đó kỹ thuật Hash Map đếm tần suất hoặc lưu vị trí đầu tiên có thể áp dụng tùy yêu cầu.

In [14]:
def count_intervals_with_score(hours, k):
    # Biến thể: ĐẾM số đoạn có tổng điểm đúng bằng k.
    freq = {0: 1}
    score = ans = 0
    for h in hours:
        score += 1 if h > 8 else -1
        ans += freq.get(score-k, 0)
        freq[score] = freq.get(score, 0) + 1
    return ans

assert count_intervals_with_score([9,8,9], 1) == 3
print('Biến thể đếm đoạn có score=k: PASS')

Biến thể đếm đoạn có score=k: PASS


## 20. Năm bài tập tự luyện — làm trước khi xem đáp án

**Bài 1 — Tính tay.** Với `hours=[8,9,9,7,9]`: viết mảng ±1, `prefix`, và tính đáp án dài nhất.

**Bài 2 — Dự đoán output.** `longestWPI([9,8,8,9,9])` trả gì? Vì sao không thể chỉ kiểm tra toàn bộ mảng?

**Bài 3 — TODO Prefix Sum.** Viết `exercise_prefix_scores(hours)` trả về `[0,p1,...,pn]`.

**Bài 4 — TODO thuật toán chính.** Tự cài `exercise_longest_wpi(hours)` bằng `dict`, không gọi lời giải có sẵn.

**Bài 5 — Biến thể.** Viết `exercise_has_positive_interval(hours, min_len)` trả `True` nếu tồn tại một đoạn hợp lệ dài **ít nhất** `min_len`, bằng cách tái sử dụng ý tưởng bài toán.

Hãy chủ động làm vào các cell phía dưới. **Không gọi TODO khi chạy notebook toàn bộ**; test chỉ bật sau khi viết xong.

In [15]:
def exercise_prefix_scores(hours):
    # TODO: khởi tạo [0], duyệt hours, append giá trị mới
    pass

def exercise_longest_wpi(hours):
    # TODO: tạo dict first; cập nhật score, best; giữ first occurrence
    pass

def exercise_has_positive_interval(hours, min_len):
    # TODO: có thể dùng độ dài dài nhất để suy ra True/False
    pass

# Chỉ mở khi đã giải TODO:
# assert exercise_prefix_scores([9,8,9]) == [0,1,0,1]
# assert exercise_longest_wpi([9,9,6,0,6,6,9]) == 3
# assert exercise_has_positive_interval([9,8,9],3) is True

## 21. Gợi ý 3 cấp độ (chưa phải đáp án)

| Bài | Gợi ý 1 — tư duy | Gợi ý 2 — công cụ | Gợi ý 3 — gần code |
|---|---|---|---|
| 1 | Mệt là +1 | Đúng 8 giờ là −1 | `prefix[k+1] = prefix[k] + delta` |
| 2 | Có thể chỉ xét đoạn con | Đếm tổng ±1 | Hỏi có đoạn dài 5 với tổng > 0 không? |
| 3 | Bắt đầu tổng 0 | `append` vào list | `p.append(p[-1]+...)` |
| 4 | Cần một tổng nhỏ hơn | `dict` lưu vị trí sớm nhất | Khi `score<=0`, tra `score-1` |
| 5 | Đề chỉ hỏi có tồn tại | Dài nhất đủ là được | `longest_wpi_hashmap(hours) >= min_len` |

## 22. Đáp án tham khảo — mở sau khi tự làm

**Bài 1:** ±1 = `[-1,+1,+1,-1,+1]`; prefix = `[0,-1,0,1,0,1]`. Cả đoạn dài 5 có tổng 1 nên đáp án **5**.

**Bài 2:** ±1 = `[+1,-1,-1,+1,+1]` có tổng 1 ⇒ toàn bộ 5 ngày hợp lệ; đáp án **5**. Việc kiểm tra toàn bộ mảng đúng ở *ví dụ này* nhưng không đủ cho mọi input (như `[9,9,6,0,6,6,9]`).

Các bài còn lại có code đáp án bên dưới. Đọc lời giải rồi thử **đóng notebook và tự viết lại**.

In [16]:
def answer_prefix_scores(hours):
    p = [0]
    for h in hours:
        p.append(p[-1] + (1 if h > 8 else -1))
    return p

def answer_longest_wpi(hours):
    first, score, best = {}, 0, 0
    for i, h in enumerate(hours):
        score += 1 if h > 8 else -1
        if score > 0:
            best = i + 1
        elif score - 1 in first:
            best = max(best, i - first[score-1])
        if score not in first:
            first[score] = i
    return best

def answer_has_positive_interval(hours, min_len):
    return answer_longest_wpi(hours) >= min_len

assert answer_prefix_scores([9,8,9]) == [0,1,0,1]
assert answer_longest_wpi([9,9,6,0,6,6,9]) == 3
assert answer_has_positive_interval([9,8,9],3) is True
assert answer_has_positive_interval([6,6,6],1) is False
print('Đáp án tham khảo 3–5: PASS')

Đáp án tham khảo 3–5: PASS


## 23. Template tư duy + Checklist submit

```text
Nếu đề nói “A nhiều hơn B” trên đoạn liên tiếp:
A → +1; B → -1
đoạn hợp lệ ⇔ tổng đoạn > 0
nếu tổng tiền tố hiện tại > 0 → lấy cả tiền tố
nếu không → tra vị trí sớm nhất của score-1
```

- [ ] Tôi hiểu `8` thuộc nhóm không mệt.
- [ ] Tôi phân biệt **đoạn liên tiếp** với chọn ngày tùy ý.
- [ ] Tôi hiểu `prefix[j]-prefix[i]` cho đoạn `[i..j-1]`.
- [ ] Tôi giải thích được vì sao dùng **first occurrence**.
- [ ] Tôi có thể tự tính `score` và `best` trên ví dụ.
- [ ] Tôi trả `int`, không trả `list`.
- [ ] Tôi test `n=1`, toàn không mệt, toàn mệt, ngày đúng 8 giờ.